# 04. Agent Progress UX — custom 스트리밍

> Part 04가 `StateGraph` 스트리밍 API 자체를 다뤘다면, 이 노트북은 LangChain V1 `create_agent`에서 **장시간 도구 실행의 사용자-facing 진행 상황**을 설계하는 데 집중합니다. 핵심은 `ToolRuntime.stream_writer`와 `stream_mode="custom"`입니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `updates` / `messages`는 Part 04에서 배운 기본 스트리밍 모드로 구분하고, 이 노트북에서는 필요한 만큼만 재사용할 수 있어요
2. `ToolRuntime.stream_writer`로 도구 실행 중 커스텀 진행 이벤트를 전송할 수 있어요
3. `stream_mode="custom"`으로 도구 내부 진행률, 단계 로그, 중간 결과를 수신할 수 있어요
4. `stream_mode=["updates", "custom"]`로 에이전트 노드 진행과 도구 내부 진행을 분리해 UI에 표시할 수 있어요

## 사전 지식

- `04_LangGraph_Advanced/07-Streaming-Steps.ipynb`: `values` / `updates` / `messages`, `subgraphs=True`, `astream_events()` 기초
- `02-Tools-V1.ipynb`: `ToolRuntime`, 도구 실행 런타임 표면
- `01-Create-Agent.ipynb`: `create_agent` 기반 에이전트 루프


## 이 노트북의 소유 범위

이 노트북은 “V1에서 스트리밍 모드가 새로 바뀌었다”를 가르치는 교재가 아니에요. `values` / `updates` / `messages`, 동기/비동기, metadata 필터링, `subgraphs=True`는 **`04_LangGraph_Advanced/07-Streaming-Steps.ipynb`가 소유**합니다.

여기서는 그 기초를 바탕으로, 실제 에이전트 앱에서 사용자가 기다리는 동안 무엇을 보여줄지 설계합니다.

| 관심사 | 소유 교재 | 이 노트북에서의 처리 |
|---|---|---|
| `values`, `updates`, `messages` 기본 차이 | `04_LangGraph_Advanced/07-Streaming-Steps.ipynb` | 다시 설명하지 않고 필요 시 연결만 함 |
| `subgraphs=True`, `astream_events()`, 태그 필터링 | `04_LangGraph_Advanced/07-Streaming-Steps.ipynb` | 이 노트북에서는 다루지 않음 |
| 도구 내부 진행률, 단계 로그, 중간 결과 | **이 노트북** | `ToolRuntime.stream_writer` + `custom` 스트림으로 구현 |
| 에이전트 단계와 도구 내부 이벤트 동시 표시 | **이 노트북** | `stream_mode=["updates", "custom"]`로 구현 |

> 🔑 **핵심 개념**: `updates`는 “에이전트 그래프의 노드가 끝났다”는 이벤트이고, `custom`은 “도구 내부에서 지금 이만큼 진행됐다”는 이벤트예요. 사용자-facing 진행률은 보통 `custom`이 담당합니다.


## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY 등을 불러와요
from dotenv import load_dotenv

load_dotenv()
# 환경 변수 로드 완료

True

In [2]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택)
# ---------------------------------------------------
# 스트리밍 과정을 LangSmith에서 시각적으로 확인할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Streaming"

## 1. 빠른 연결: `create_agent`도 같은 stream API를 쓴다

`create_agent`는 내부적으로 LangGraph 런타임을 사용하므로 `agent.stream(...)` 인터페이스를 그대로 공유해요. 하지만 여기서 `updates`와 `messages`를 다시 실습하지는 않습니다.

- 노드 완료 추적이 필요하면 `stream_mode="updates"`
- 최종 응답 토큰을 채팅 UI에 흘리고 싶으면 `stream_mode="messages"`
- **도구 내부 진행률을 직접 보내고 싶으면 `stream_mode="custom"`**

이제부터는 세 번째 경우, 즉 `custom` 이벤트 설계에 집중합니다.


In [3]:
# ---------------------------------------------------
# Agent Progress UX 예제 공통 import
# ---------------------------------------------------
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime

# 기본 모델: gpt-4o-mini (비용 효율적)
model = init_chat_model("openai:gpt-4o-mini")


## 2. 핵심: `custom` 모드와 `ToolRuntime.stream_writer`

`stream_mode="custom"`은 도구(Tool) 실행 중에 도구가 직접 보낸 이벤트를 수신하는 모드예요. V1 도구는 `runtime: ToolRuntime`을 매개변수로 받을 수 있고, 이 객체의 `stream_writer`를 호출해 진행 상황, 단계 로그, 중간 결과 등을 외부로 전송할 수 있어요.

**사용법:**
1. 도구 함수 매개변수에 `runtime: ToolRuntime`을 선언해요. 이 값은 모델에게 노출되지 않고 런타임이 자동 주입해요.
2. 도구 내부에서 `runtime.stream_writer({...})`를 호출해요.
3. 에이전트를 `stream_mode="custom"`으로 실행해 도구가 보낸 이벤트를 받아요.

> 💡 **실무 팁**: 커스텀 이벤트는 문자열보다 딕셔너리로 보내는 편이 좋아요. `type`, `stage`, `progress`, `message` 같은 키를 정해두면 프론트엔드에서 진행률 바, 타임라인, 로그 뷰로 안정적으로 렌더링할 수 있어요.


In [4]:
# ---------------------------------------------------
# ToolRuntime.stream_writer를 사용하는 도구 정의
# ---------------------------------------------------
@tool
def process_data(data_size: int, runtime: ToolRuntime) -> str:
    """Process data with real-time progress updates."""
    writer = runtime.stream_writer
    total = max(data_size, 1)

    # 도구 내부 진행 이벤트 스키마를 일정하게 유지해요.
    writer({
        "type": "progress",
        "stage": "start",
        "processed": 0,
        "total": total,
        "message": "데이터 처리를 시작합니다.",
    })

    # 10개 단위로 처리하며 진행 상황을 전송해요.
    for i in range(0, total, 10):
        processed = min(i + 10, total)
        writer({
            "type": "progress",
            "stage": "processing",
            "processed": processed,
            "total": total,
            "message": f"{processed}/{total}개 처리 완료",
        })

    writer({
        "type": "progress",
        "stage": "done",
        "processed": total,
        "total": total,
        "message": "데이터 처리가 완료되었습니다.",
    })

    return f"{total}개 데이터 처리 완료!"


agent_custom = create_agent(model=model, tools=[process_data])


In [5]:
# ---------------------------------------------------
# custom 모드로 도구 내부 진행 상황 수신
# ---------------------------------------------------
inputs = {"messages": [{"role": "user", "content": "50개의 데이터를 처리해줘"}]}

print()

for event in agent_custom.stream(inputs, stream_mode="custom"):
    # event는 stream_writer()로 전송한 데이터 그대로예요.
    if isinstance(event, dict) and event.get("type") == "progress":
        pct = (event["processed"] / event["total"]) * 100
        bar_len = 20
        filled = int(bar_len * event["processed"] / event["total"])
        bar = "█" * filled + "░" * (bar_len - filled)
        print(f"\r진행: [{bar}] {pct:>5.1f}% - {event['message']}", end="", flush=True)

print("\n완료")



진행: [████████████████████] 100.0% - 데이터 처리가 완료되었습니다.
완료


## 3. 에이전트 단계와 도구 내부 이벤트를 함께 보기

사용자에게는 두 종류의 진행 정보가 필요할 때가 많아요.

1. **에이전트 단계**: 모델이 도구를 고르고, 도구가 끝나고, 다시 모델이 응답하는 큰 흐름
2. **도구 내부 단계**: 도구가 데이터 10%, 20%, 30%를 처리하는 세부 흐름

`stream_mode=["updates", "custom"]`처럼 리스트를 전달하면 두 이벤트를 한 스트림에서 함께 받을 수 있어요. 이때 반환 형식은 `(mode_name, data)` 튜플입니다.

```python
stream_mode=["updates", "custom"]
# 반환: ("updates", {...}) 또는 ("custom", {...})
```


In [6]:
# ---------------------------------------------------
# 다중 스트리밍 모드: updates + custom 동시 사용
# ---------------------------------------------------
inputs = {"messages": [{"role": "user", "content": "30개의 데이터를 처리해줘"}]}

print()

for mode, data in agent_custom.stream(inputs, stream_mode=["updates", "custom"]):
    if mode == "updates":
        # 에이전트 그래프의 큰 단계 완료 이벤트
        for node_name in data.keys():
            print(f"[에이전트 단계 완료] {node_name}")

    elif mode == "custom":
        # 도구 내부에서 직접 보낸 세부 진행 이벤트
        if isinstance(data, dict) and data.get("type") == "progress":
            print(f"  └ [도구 진행] {data['message']}")



[에이전트 단계 완료] model
  └ [도구 진행] 데이터 처리를 시작합니다.
  └ [도구 진행] 10/30개 처리 완료
  └ [도구 진행] 20/30개 처리 완료
  └ [도구 진행] 30/30개 처리 완료
  └ [도구 진행] 데이터 처리가 완료되었습니다.
[에이전트 단계 완료] tools
[에이전트 단계 완료] model


> ⚠️ **자주 하는 실수**: 단일 모드와 다중 모드의 반환 형식이 달라요.
>
> - 단일 모드 `stream_mode="custom"`: `event` 자체가 도구가 보낸 데이터
> - 다중 모드 `stream_mode=["updates", "custom"]`: `(mode, data)` 튜플
>
> UI 어댑터를 만들 때는 먼저 단일 모드인지 다중 모드인지 결정하고, 반환 형식을 한 곳에서 정규화하는 편이 좋아요.


## 4. 종합 예제: 단계별 진행 보고 에이전트

이번에는 데이터 분석 도구가 여러 단계를 실행하면서 `custom` 이벤트를 보내고, 에이전트 전체 흐름은 `updates`로 함께 관찰해요. 이 예제의 핵심은 스트리밍 API 복습이 아니라 **진행 이벤트 스키마를 설계하고 UI가 읽기 쉬운 형태로 보내는 것**입니다.

> 🎯 **강의 포인트**: 장시간 실행 AI 파이프라인에서는 “모델이 생각 중”이라는 막연한 표시보다 “데이터 로딩 → 정제 → 분석 → 리포트 생성”처럼 단계별 상태를 보여주는 것이 사용자 신뢰를 높여요.


In [7]:
# ---------------------------------------------------
# 종합 예제: 단계별 진행 보고 에이전트
# ---------------------------------------------------
import time


@tool
def analyze_sales_data(dataset_name: str, num_records: int, runtime: ToolRuntime) -> str:
    """Analyze sales data with step-by-step progress reporting."""
    writer = runtime.stream_writer

    steps = [
        ("loading", "데이터 로딩"),
        ("cleaning", "데이터 정제"),
        ("processing", "통계 분석"),
        ("reporting", "리포트 생성"),
    ]

    total_steps = len(steps)
    for idx, (step_name, step_desc) in enumerate(steps, start=1):
        writer({
            "type": "step",
            "stage": step_name,
            "status": "start",
            "current": idx,
            "total": total_steps,
            "message": f"{step_desc} 시작",
        })
        time.sleep(0.2)
        writer({
            "type": "step",
            "stage": step_name,
            "status": "done",
            "current": idx,
            "total": total_steps,
            "message": f"{step_desc} 완료",
        })

    return f"{dataset_name} 데이터셋 {num_records:,}개 레코드 분석 완료!"


comp_agent = create_agent(model=model, tools=[analyze_sales_data])

inputs = {
    "messages": [
        {"role": "user", "content": "영업 데이터셋 1000개 레코드를 분석해줘"}
    ]
}

print()

for mode, data in comp_agent.stream(inputs, stream_mode=["updates", "custom"]):
    if mode == "custom":
        if isinstance(data, dict) and data.get("type") == "step":
            icon = "✓" if data["status"] == "done" else "→"
            print(f"  {icon} ({data['current']}/{data['total']}) {data['message']}")

    elif mode == "updates":
        for node_name, node_data in data.items():
            if "messages" in node_data:
                last_msg = node_data["messages"][-1]
                content = last_msg.content
                if isinstance(content, str) and content:
                    print(f"\n[{node_name}] 최종 응답:")
                    print(content)

print()



  → (1/4) 데이터 로딩 시작
  ✓ (1/4) 데이터 로딩 완료
  → (2/4) 데이터 정제 시작
  ✓ (2/4) 데이터 정제 완료
  → (3/4) 통계 분석 시작
  ✓ (3/4) 통계 분석 완료
  → (4/4) 리포트 생성 시작
  ✓ (4/4) 리포트 생성 완료

[tools] 최종 응답:
영업 데이터셋 데이터셋 1,000개 레코드 분석 완료!

[model] 최종 응답:
영업 데이터셋 1,000개 레코드의 분석이 완료되었습니다! 추가로 알고 싶은 정보가 있으면 말씀해 주세요.



## 실습: 다중 도구 진행 상황 추적기

아래 코드를 완성하여 두 개의 도구를 순서대로 사용하는 에이전트를 만들어보세요. 각 도구는 동일한 이벤트 스키마로 진행 상황을 `custom` 스트림에 보내고, 전체 흐름은 `updates + custom` 다중 모드로 추적해요.


In [8]:
# ============================================================
# 실습 해설: 다중 도구 진행 상황 추적기 완성하기
# ============================================================

import time


@tool
def fetch_data(num_items: int, runtime: ToolRuntime) -> str:
    """Fetch data items one by one with progress updates."""
    writer = runtime.stream_writer

    for i in range(num_items):
        writer({
            "type": "tool_progress",
            "tool": "fetch_data",
            "current": i + 1,
            "total": num_items,
            "message": f"{i + 1}/{num_items} 항목 로드",
        })
        time.sleep(0.1)

    return f"{num_items}개 항목 로드 완료"


@tool
def summarize_data(content: str, runtime: ToolRuntime) -> str:
    """Summarize the data in 3 stages."""
    writer = runtime.stream_writer

    stages = ["텍스트 분석", "키워드 추출", "요약 생성"]
    for idx, stage in enumerate(stages, start=1):
        writer({
            "type": "tool_progress",
            "tool": "summarize_data",
            "current": idx,
            "total": len(stages),
            "message": f"{stage} 완료",
        })
        time.sleep(0.1)

    return f"'{content}' 요약 완료"


multi_agent = create_agent(model=model, tools=[fetch_data, summarize_data])

for mode, data in multi_agent.stream(
    {"messages": [{"role": "user", "content": "데이터 5개를 가져오고 요약해줘"}]},
    stream_mode=["updates", "custom"],
):
    if mode == "custom":
        print("[진행]", data)
    elif mode == "updates":
        print("[업데이트]", list(data.keys()))


[업데이트] ['model']
[진행] {'type': 'tool_progress', 'tool': 'fetch_data', 'current': 1, 'total': 5, 'message': '1/5 항목 로드'}
[진행] {'type': 'tool_progress', 'tool': 'fetch_data', 'current': 2, 'total': 5, 'message': '2/5 항목 로드'}
[진행] {'type': 'tool_progress', 'tool': 'fetch_data', 'current': 3, 'total': 5, 'message': '3/5 항목 로드'}
[진행] {'type': 'tool_progress', 'tool': 'fetch_data', 'current': 4, 'total': 5, 'message': '4/5 항목 로드'}
[진행] {'type': 'tool_progress', 'tool': 'fetch_data', 'current': 5, 'total': 5, 'message': '5/5 항목 로드'}
[업데이트] ['tools']
[업데이트] ['model']
[진행] {'type': 'tool_progress', 'tool': 'summarize_data', 'current': 1, 'total': 3, 'message': '텍스트 분석 완료'}
[진행] {'type': 'tool_progress', 'tool': 'summarize_data', 'current': 2, 'total': 3, 'message': '키워드 추출 완료'}
[진행] {'type': 'tool_progress', 'tool': 'summarize_data', 'current': 3, 'total': 3, 'message': '요약 생성 완료'}
[업데이트] ['tools']
[업데이트] ['model']


## 진행 이벤트 설계 가이드

이 노트북의 핵심은 “어떤 모드를 외우느냐”보다, 사용자에게 보여줄 이벤트를 어떻게 설계하느냐예요.

| 상황 | 권장 방식 | 이유 |
|------|----------|------|
| 최종 답변 토큰을 채팅 UI에 표시 | `messages` | Part 04에서 배운 토큰 스트림을 그대로 사용 |
| 에이전트가 어느 노드를 지나는지 표시 | `updates` | `model`, `tools` 같은 큰 단계 완료를 알 수 있음 |
| 장시간 도구 진행률 표시 | `custom` | 도구가 직접 세부 이벤트를 보낼 수 있음 |
| 단계 로그 + 진행률 바 + 최종 응답 | `stream_mode=["updates", "custom"]` + 필요 시 `messages` | 큰 흐름과 도구 내부 흐름을 분리 가능 |

### 커스텀 이벤트 스키마 예시

```python
{
    "type": "progress",
    "stage": "processing",
    "current": 3,
    "total": 10,
    "message": "3/10개 처리 완료",
}
```

> 💡 **실무 팁**: 프론트엔드와 백엔드가 같은 이벤트 스키마를 공유하면, 진행률 바·타임라인·로그 패널을 같은 스트림에서 안정적으로 만들 수 있어요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **역할 분리**: `values` / `updates` / `messages`, `subgraphs=True`, `astream_events()` 기본기는 `04_LangGraph_Advanced/07-Streaming-Steps.ipynb`가 소유해요
- **이 노트북의 소유 개념**: V1 `create_agent` 도구가 `ToolRuntime.stream_writer`로 사용자-facing 진행 이벤트를 보내는 패턴이에요
- **`stream_mode="custom"`**: 도구 내부 진행률, 단계 로그, 중간 결과를 외부로 보낼 수 있어요
- **다중 모드**: `stream_mode=["updates", "custom"]`를 사용하면 에이전트 노드 완료와 도구 내부 이벤트를 분리해 렌더링할 수 있어요
- **이벤트 스키마**: `type`, `stage`, `current`, `total`, `message`처럼 UI가 읽기 쉬운 키를 정해두는 것이 중요해요


## 다음 노트북 예고

다음 `05-Runtime-Context.ipynb`에서는 **Runtime 객체, config, context 전달**을 배워요. 여기서 사용한 `ToolRuntime`을 더 깊이 이해하고, 에이전트 실행 중 사용자별 설정과 장기 메모리에 접근하는 방법을 알아볼게요.
